# Personalised Inflation Predictor

This notebook documents and demonstrates the `inflation_predictor.py` module, which generates **household-level month-over-month (MoM) inflation predictions** for Canadian households based on their spending profile, demographics, and tenure status.

---

## 1 · Background and Motivation

Statistics Canada's Consumer Price Index (CPI) measures the average price change experienced by a *representative* consumer basket. But Canadian households differ substantially in what they buy: a renting household in their 30s spends very little on vehicle purchases and more on food away from home, while a retired homeowner may spend more on health care and utilities. These spending differences mean that **aggregate CPI is not the inflation rate actually experienced by any individual household**.

This project constructs **personalised inflation rates** by:

1. Using the **Survey of Household Spending (SHS)** (Statistics Canada microdata, waves 2017 / 2019 / 2021 / 2023) to measure each household's actual spending mix across CPI major categories.
2. **CC-weighting** the spending basket: each category's spending is multiplied by the share of that category paid by credit card or mobile (drawn from the Bank of Canada Methods-of-Payment Survey), so the basket reflects only *traceable digital transactions*.
3. Applying the CPI major-component month-over-month growth rates to each household's CC-weighted basket shares to derive a household-specific inflation rate:

$$
\pi_{h,t} = \sum_i w_{i,h} \cdot \pi_{i,t} \;+\; \left(1 - \sum_i w_{i,h}\right) \cdot \pi_{\text{all-items},t}
$$

where $w_{i,h}$ is household $h$'s CC-weighted share of spending on component $i$, and $\pi_{i,t}$ is the MoM CPI growth rate for component $i$ in month $t$.

The remaining share $(1 - \sum_i w_{i,h})$ — spending not captured by the mapped CPI categories — is inflated at the all-items CPI rate.

---

## 2 · The Spending Basket

### Denominator

The basket denominator is **total consumption minus shelter, vehicle purchases, and recreational vehicles**:

$$\text{Basket} = TC001 - SH001 - TR004 - RV020$$

Shelter (SH001) is excluded because the CPI uses a rental-equivalence / mortgage-interest concept that differs from SHS actual outlays. Vehicle purchases (TR004) and recreational vehicles (RV020) are large lumpy capital expenditures that CPI does not capture as a flow of services.

### CC-Weighted Numerators

Each CPI-mapped spending category is scaled by its credit/mobile payment share $c_i$ before entering the basket:

| SHS category | CPI major | $c_i$ |
|---|---|---|
| Food (FD001) | Food | 0.539 |
| Tobacco & alcohol (TA018 …) | Alcoholic bev., tobacco … | 0.539 |
| Household furnishings (HF001) | HH operations, furnishings … | 0.851 |
| Household operations (HO001) | HH operations, furnishings … | 0.620 |
| Clothing & footwear (CC001) | Clothing and footwear | 0.798 |
| Transportation (TR001 − TR004) | Transportation | 0.648 |
| Health care (HC001) | Health and personal care | 0.659 |
| Personal care (PC001) | Health and personal care | 0.620 |
| Recreation (RE001 − RV020) | Recreation, education … | 0.747 |
| Education (ED002) | Recreation, education … | 0.590 |

The CC-weighted bottom-up spending denominator is then:

$$CCspend_{h} = \sum_i c_i \cdot \max(spend_{i,h} - excl_{i,h}, 0)$$

Households are ranked into deciles ($D_1$ = lowest, $D_{10}$ = highest) by $CCspend_h$ using the SHS survey weights.

---

## 3 · The Regression Models

Having computed $\pi_{h,t}$ for each household $h$ and each calendar month $t$, we fit predictive regressions using three household characteristics that are observable without microdata:

- **`CCspend_decilegrp`** — CC-spending decile ($D_1$–$D_{10}$, reference $D_1$)
- **`age_group`** — reference person's age group (reference: *0–29*)
- **`has_mortgage`** — *Mortgage* vs *No mortgage* (tenure == 1 in SHS)

All regressions use SHS survey weights.

### Option A — One model per calendar month

For each month $t$ independently:

$$\pi_{h,t} = \beta_{0,t} + \beta_{\text{decile},t}\, D_{h} + \beta_{\text{age},t}\, A_{h} + \beta_{\text{mort},t}\, M_{h} + \varepsilon_{h,t}$$

This yields ~84 separate weighted OLS models (Jan 2019 – present). Each model is cross-sectional: the same households appear in every month with *identical covariates* but different $\pi_{h,t}$. The R² in any given month is therefore driven by cross-sectional heterogeneity in spending mix, not time variation. Months with large dispersion in component CPI growth rates (e.g. the 2022 energy spike) tend to show higher R².

### Option B — Pooled panel with month fixed effects

Stack all household × month observations into one model:

$$\pi_{h,t} = \alpha_t + \beta_{\text{decile}}\, D_{h} + \beta_{\text{age}}\, A_{h} + \beta_{\text{mort}}\, M_{h} + \varepsilon_{h,t}$$

The month fixed effects $\alpha_t$ absorb the aggregate CPI movement common to all households. The structural coefficients $\beta$ then identify how spending-mix and demographic heterogeneity *amplify or dampen* the common aggregate shock.

**Standard errors are clustered at the household level** (using `sandwich::vcovCL` in R), because each household's covariates are identical across all months, inducing perfect within-household correlation in the residuals.

### Choosing between A and B

| | Option A | Option B |
|---|---|---|
| Coefficients vary by | month | fixed (one set) |
| Month effect | baked into intercept | explicit FE (estimable) |
| SE correction | none needed (cross-section) | clustered by household |
| Best for | episode-specific questions | distributional structure |
| Prediction inputs | year + month + demographics | year + month + demographics |

---

## 4 · File Prerequisites

The tables below were estimated from SHS microdata (2021 basket vintage) and are stored in `outputs/inflation_predictor/`. The estimation script stays local because it reads restricted survey files.

| File | Contents |
|---|---|
| `monthly_infl_reg_coefs_A_v21.csv` | Coefficients for every monthly Option A model |
| `monthly_infl_reg_fit_A_v21.csv` | R², RMSE, mean actual/fitted per month |
| `monthly_infl_reg_coefs_B_structural_v21.csv` | Structural coefficients (clustered SEs) |
| `monthly_infl_reg_coefs_B_month_fe_v21.csv` | Month fixed-effect coefficients |
| `monthly_infl_reg_fit_B_v21.csv` | Pooled model fit statistics |

The coefficient tables are committed under `outputs/inflation_predictor/`. `OUTDIR` below resolves that folder from the repository root.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "outputs" / "inflation_predictor").is_dir() and (
            candidate / "code" / "inflation_predictor"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the relative_inflation repository."
    )

OUTDIR  = _repo_root() / "outputs" / "inflation_predictor"
VINTAGE = "v21"   # '2021 SHS basket'

# Check that the required CSVs are present
REQUIRED = [
    f"monthly_infl_reg_coefs_A_{VINTAGE}.csv",
    f"monthly_infl_reg_fit_A_{VINTAGE}.csv",
    f"monthly_infl_reg_coefs_B_structural_{VINTAGE}.csv",
    f"monthly_infl_reg_coefs_B_month_fe_{VINTAGE}.csv",
    f"monthly_infl_reg_fit_B_{VINTAGE}.csv",
]

missing = [f for f in REQUIRED if not (OUTDIR / f).exists()]
if missing:
    warnings.warn(
        "The following CSV files are missing from outputs/inflation_predictor:\n"
        + "\n".join(f"  {OUTDIR / f}" for f in missing)
    )
else:
    print("✓ All required CSV files found.")

---

## 5 · Loading the Predictors

In [ ]:
import sys
sys.path.insert(0, str(_repo_root() / "code" / "inflation_predictor"))

from inflation_predictor import load_predictors

pred_a, pred_b = load_predictors(OUTDIR, vintage=VINTAGE)

months_a = pred_a.available_months()
print(f"Option A  →  {len(months_a)} monthly models  "
      f"({months_a[0]} … {months_a[-1]})")

months_b = pred_b.available_months()
print(f"Option B  →  {len(months_b)} month FE coefficients  "
      f"({months_b[0]} … {months_b[-1]})")

---

## 6 · Input Schema

Both predictors accept a `pd.DataFrame` with the following columns.

| Column | dtype | Valid values |
|---|---|---|
| `year` | int | e.g. 2019–2026 |
| `month` | int | 1–12 |
| `CCspend_decilegrp` | str | `"D1"` (lowest CC spenders) … `"D10"` |
| `age_group` | str | `"0-29"`, `"30-39"`, `"40-54"`, `"55-64"`, `"65-74"`, `"75+"` |
| `has_mortgage` | str | `"No mortgage"` or `"Mortgage"` |

The **reference categories** (absorbed into the intercept) are `D1`, `0-29`, and `No mortgage`.

> **Option A** returns `NaN` for any row whose year-month has no fitted model (e.g. months before 2019 or after the latest available CPI release).
>
> **Option B** always returns a value; months with no estimated FE use `α_t = 0` (the prediction reduces to the structural component only).

---

## 7 · Point Predictions

In [ ]:
sample = pd.DataFrame([
    # Typical high-inflation month (Jun 2022 — peak CPI period)
    dict(year=2022, month=6,  CCspend_decilegrp="D1",  age_group="0-29",  has_mortgage="No mortgage"),
    dict(year=2022, month=6,  CCspend_decilegrp="D5",  age_group="40-54", has_mortgage="Mortgage"),
    dict(year=2022, month=6,  CCspend_decilegrp="D10", age_group="65-74", has_mortgage="No mortgage"),
    # A moderate month
    dict(year=2023, month=3,  CCspend_decilegrp="D3",  age_group="30-39", has_mortgage="Mortgage"),
    dict(year=2023, month=3,  CCspend_decilegrp="D7",  age_group="55-64", has_mortgage="No mortgage"),
    # Near the start of the estimation window
    dict(year=2019, month=1,  CCspend_decilegrp="D2",  age_group="75+",   has_mortgage="No mortgage"),
])

sample["pred_A"] = pred_a.predict(sample)
sample["pred_B"] = pred_b.predict(sample)

print("Predicted MoM personalised inflation (percentage points)")
print(sample.to_string(index=False, float_format="{:.4f}".format))

---

## 8 · Uncertainty Bounds

Both predictors expose a `predict_with_uncertainty()` method that returns approximate 95 % confidence intervals.

**Important caveat**: the SE approximation uses the *diagonal* of the coefficient variance-covariance matrix (i.e. root-sum-of-squared per-term SEs). This ignores covariance between coefficients and is therefore conservative.

In [ ]:
coef_a_path = OUTDIR / f"monthly_infl_reg_coefs_A_{VINTAGE}.csv"

unc_a = pred_a.predict_with_uncertainty(sample, se_path=coef_a_path)
unc_b = pred_b.predict_with_uncertainty(sample, include_fe_se=True)

print("── Option A: predictions ± 95 % CI ──")
print(unc_a.round(4).to_string())

print("\n── Option B: predictions ± 95 % CI (clustered structural + FE SE) ──")
print(unc_b.round(4).to_string())

---

## 9 · Decile Sweep: predicted inflation across the spending distribution

Fix a calendar month and demographic profile, then vary the CC-spending decile to see how inflation dispersion relates to spending intensity.

In [ ]:
SWEEP_YEAR  = 2022
SWEEP_MONTH = 6
SWEEP_AGE   = "40-54"
SWEEP_MORT  = "No mortgage"

sweep = pd.DataFrame([
    dict(year=SWEEP_YEAR, month=SWEEP_MONTH,
         CCspend_decilegrp=f"D{d}",
         age_group=SWEEP_AGE, has_mortgage=SWEEP_MORT)
    for d in range(1, 11)
])

sweep["pred_A"] = pred_a.predict(sweep)
sweep["pred_B"] = pred_b.predict(sweep)

print(f"Predicted MoM inflation  –  {SWEEP_YEAR}-{SWEEP_MONTH:02d}, "
      f"age {SWEEP_AGE}, {SWEEP_MORT}")
print(sweep[["CCspend_decilegrp", "pred_A", "pred_B"]].to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = range(1, 11)
ax.plot(x, sweep["pred_A"], marker="o", label="Option A", color="#2166ac")
ax.plot(x, sweep["pred_B"], marker="s", label="Option B", color="#d73027", linestyle="--")

ax.set_xticks(range(1, 11))
ax.set_xticklabels([f"D{d}" for d in range(1, 11)])
ax.set_xlabel("CC-spend decile  (D1 = lowest, D10 = highest)")
ax.set_ylabel("Predicted MoM inflation (pp)")
ax.set_title(f"Predicted personalised inflation by spending decile\n"
             f"{SWEEP_YEAR}-{SWEEP_MONTH:02d}  ·  age {SWEEP_AGE}  ·  {SWEEP_MORT}")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---

## 10 · Option A — R² over time

The month-by-month R² shows *which episodes produced the greatest spending-mix driven inflation dispersion*. High R² months correspond to periods when CPI components diverged sharply (e.g. energy vs food vs services).

In [ ]:
fit_a = pd.read_csv(OUTDIR / f"monthly_infl_reg_fit_A_{VINTAGE}.csv",
                    parse_dates=["ref_date"])

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

# R²
ax = axes[0]
ax.plot(fit_a["ref_date"], fit_a["r_squared"], color="#2166ac", linewidth=0.9)
ax.fill_between(fit_a["ref_date"], 0, fit_a["r_squared"],
                alpha=0.15, color="#2166ac")
ax.set_ylabel("R²")
ax.set_title("Option A — monthly cross-sectional R² and RMSE  (2021 SHS vintage)")
ax.grid(axis="y", alpha=0.3)

# RMSE
ax = axes[1]
ax.plot(fit_a["ref_date"], fit_a["rmse_weighted"], color="#d73027", linewidth=0.9)
ax.set_ylabel("Weighted RMSE (pp)")
ax.set_xlabel("Month")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

---

## 11 · Option A — Decile coefficients over time

Each line shows how much higher (or lower) MoM inflation was experienced by households in that CC-spending decile *relative to D1* in a given month. Positive divergence at peak inflation months indicates that higher-spending households faced more inflation pressure.

In [ ]:
coef_a = pd.read_csv(OUTDIR / f"monthly_infl_reg_coefs_A_{VINTAGE}.csv",
                     parse_dates=["ref_date"])

decile_coefs = coef_a[coef_a["term"].str.startswith("CCspend_decilegrp")].copy()
decile_coefs["decile"] = decile_coefs["term"].str.replace("CCspend_decilegrp", "", regex=False)

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.cm.RdYlBu_r
deciles = sorted(decile_coefs["decile"].unique(),
                 key=lambda d: int(d[1:]))
colours = {d: cmap(i / (len(deciles) - 1)) for i, d in enumerate(deciles)}

for d in deciles:
    sub = decile_coefs[decile_coefs["decile"] == d].sort_values("ref_date")
    ax.plot(sub["ref_date"], sub["estimate"],
            label=d, color=colours[d], linewidth=0.85, alpha=0.9)

ax.axhline(0, color="grey", linewidth=0.6, linestyle="--")
ax.set_ylabel("Coefficient vs D1 (pp MoM)")
ax.set_xlabel("Month")
ax.set_title("Option A — CCspend decile coefficients by month  (2021 SHS vintage)\n"
             "Reference: D1.  Positive = higher MoM inflation than D1.")
ax.legend(title="Decile", loc="upper left", fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---

## 12 · Option B — Month fixed effects

The month FEs represent the aggregate MoM CPI movement absorbed by the pooled model. Plotting them recovers a weighted-average picture of aggregate inflation momentum, after removing the structural cross-sectional variation.

In [ ]:
fe_b = pd.read_csv(OUTDIR / f"monthly_infl_reg_coefs_B_month_fe_{VINTAGE}.csv")
fe_b["ref_date"] = pd.to_datetime(fe_b["ym_label"] + "-01")
fe_b = fe_b.sort_values("ref_date")

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(fe_b["ref_date"],
                fe_b["estimate"] - 1.96 * fe_b["std_error"],
                fe_b["estimate"] + 1.96 * fe_b["std_error"],
                alpha=0.2, color="#d73027")
ax.plot(fe_b["ref_date"], fe_b["estimate"], color="#d73027", linewidth=1.0)
ax.axhline(0, color="grey", linewidth=0.6, linestyle="--")

ax.set_ylabel("Month FE coefficient (pp MoM)")
ax.set_xlabel("Month")
ax.set_title("Option B — absorbed aggregate MoM CPI (month fixed effects)\n"
             "Ribbon = ±1.96 × clustered SE.  Reference month: first in window.")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---

## 13 · Option B — Structural coefficients

These are the coefficients that remain after the month FEs absorb aggregate inflation. They describe how a household's spending decile, age, and tenure predict *deviations from the aggregate MoM rate*.

In [ ]:
struct_b = pd.read_csv(OUTDIR / f"monthly_infl_reg_coefs_B_structural_{VINTAGE}.csv")

# Drop intercept for plotting; it absorbs the reference month + reference categories
plot_struct = struct_b[struct_b["term"] != "(Intercept)"].copy()
plot_struct["lo95"] = plot_struct["estimate"] - 1.96 * plot_struct["std_error"]
plot_struct["hi95"] = plot_struct["estimate"] + 1.96 * plot_struct["std_error"]
plot_struct["sig"]  = plot_struct["p_value"] < 0.05
plot_struct = plot_struct.sort_values("estimate", ascending=True)

colours = plot_struct["sig"].map({True: "#d73027", False: "#4393c3"})

fig, ax = plt.subplots(figsize=(8, max(5, len(plot_struct) * 0.35)))
y_pos = range(len(plot_struct))

ax.barh(y_pos, plot_struct["estimate"], xerr=1.96 * plot_struct["std_error"],
        color=colours, alpha=0.8, error_kw=dict(ecolor="grey", capsize=3, linewidth=0.8))
ax.axvline(0, color="grey", linewidth=0.6, linestyle="--")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_struct["term_label"], fontsize=8)
ax.set_xlabel("Coefficient (pp MoM differential vs reference, clustered SEs)")
ax.set_title("Option B — structural coefficients after absorbing month FEs\n"
             "Red = p < 0.05  ·  Blue = not significant  ·  Bars = 95 % CI")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

---

## 14 · Comparing Options A and B over a full time series

For a fixed household profile, plot both predictions side-by-side across all available months.

In [ ]:
# Build a grid of all available months × two household profiles
profiles = [
    dict(label="D1 / age 0-29 / no mortgage",
         CCspend_decilegrp="D1",  age_group="0-29",  has_mortgage="No mortgage",
         colour_a="#2166ac", colour_b="#6baed6"),
    dict(label="D10 / age 65-74 / no mortgage",
         CCspend_decilegrp="D10", age_group="65-74", has_mortgage="No mortgage",
         colour_a="#d73027", colour_b="#f4a582"),
]

all_months = pd.DataFrame({"ym_label": pred_a.available_months()})
all_months["ref_date"] = pd.to_datetime(all_months["ym_label"] + "-01")
all_months["year"]  = all_months["ref_date"].dt.year
all_months["month"] = all_months["ref_date"].dt.month

fig, ax = plt.subplots(figsize=(12, 5))

for p in profiles:
    ts = all_months.copy()
    ts["CCspend_decilegrp"] = p["CCspend_decilegrp"]
    ts["age_group"]         = p["age_group"]
    ts["has_mortgage"]      = p["has_mortgage"]

    ts["pred_A"] = pred_a.predict(ts)
    ts["pred_B"] = pred_b.predict(ts)

    ax.plot(ts["ref_date"], ts["pred_A"],
            color=p["colour_a"], linewidth=1.0, label=f"A: {p['label']}")
    ax.plot(ts["ref_date"], ts["pred_B"],
            color=p["colour_b"], linewidth=0.8, linestyle="--",
            label=f"B: {p['label']}")

ax.axhline(0, color="grey", linewidth=0.5, linestyle=":")
ax.set_ylabel("Predicted MoM inflation (pp)")
ax.set_xlabel("Month")
ax.set_title("Options A and B: predicted personalised MoM inflation for two household profiles")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---

## 15 · Interpreting predictions

- All outputs are in **percentage points, month-over-month**. To annualise approximately: multiply by 12, or compound $\prod (1 + \pi_t/100) - 1$.
- The models are estimated on the **2021 SHS vintage** (spending basket fixed at 2021 levels). The basket does not update within the prediction window, so the structural coefficients reflect 2021 spending patterns applied to any requested month.
- A parallel 2023-vintage model set is available in `monthly_infl_reg_coefs_*_v23.csv` once the R script has been run. Pass `vintage="v23"` to `load_predictors()` to use it.
- Option B's **month FE is the dominant driver** of the prediction in any given month; the structural predictors shift it by a small number of basis points. Do not interpret the structural coefficients in isolation from the FE.

In [ ]:
# Decompose an Option B prediction for a single row into its components
row = pd.DataFrame([dict(
    year=2022, month=6,
    CCspend_decilegrp="D8",
    age_group="55-64",
    has_mortgage="Mortgage"
)])

ym = f"{int(row.year.iloc[0])}-{int(row.month.iloc[0]):02d}"
fe_est, fe_se = pred_b.month_fe(ym)
total_pred    = pred_b.predict(row).iloc[0]
structural    = total_pred - fe_est

print(f"Prediction decomposition for {ym}, D8, age 55-64, Mortgage")
print(f"  Month FE (α_t)          : {fe_est:+.4f} pp  (SE = {fe_se:.4f})")
print(f"  Structural (β·X)        : {structural:+.4f} pp")
print(f"  Total prediction        : {total_pred:+.4f} pp")